In [36]:
# Launch the LAMBDA server as a background process and capture its 
# console output into the 'log_lines' list for real-time monitoring.
import subprocess
import threading
import time
from IPython.display import display, Markdown

log_lines = []

def stream_logs(proc):
    for line in iter(proc.stdout.readline, b''):
        decoded = line.decode().rstrip()
        log_lines.append(decoded)

proc = subprocess.Popen(
    ["python", "lambda_app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd='/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA'
)

t = threading.Thread(target=stream_logs, args=(proc,))
t.daemon = True
t.start()

print("LAMBDA server starting...")
time.sleep(8)
print("Server ready.")

LAMBDA server starting...
Server ready.


In [37]:
# Connect to the local LAMBDA server, upload the housing dataset, 
# and pause briefly to ensure the file is fully cached before analysis.
from gradio_client import Client, handle_file

client = Client("http://127.0.0.1:8000")
csv_path = '/Users/carter/Desktop/Prof.Zheng_Project/california_housing_affordability.csv'

client.predict(files=handle_file(csv_path), api_name="/add_file")
time.sleep(3)
print("Dataset uploaded.")

Loaded as API: http://127.0.0.1:8000/ ✔
Dataset uploaded.


In [38]:
# OVERALL PROCESS: 
# This script automates a two-stage "Agentic Workflow" where:
# 1. The Orchestrator plans the ML pipeline based on the prompt.
# 2. The Programmer/Inspector execute and verify the code.
# 3. The script monitors background server logs for specific completion signals 
#    (like 'Accuracy') to ensure the report is generated only after the agents finish.
from datetime import datetime
import yaml


prompt = (
    """
    1. Load the dataset and perform any necessary preprocessing.
    2. Perform exploratory data analysis to understand key patterns in the data.
    3. Split the data into training and test sets with a proportion of 8:2 (do not fix the random_state).
    4. Select relevant features for modeling.
    5. Train a logistic regression model and evaluate it by accuracy on the test set.
    6. Train an SVM model and evaluate it by accuracy on the test set.
    7. Train an MLP model and evaluate it by accuracy on the test set.
    8. Train a decision tree model and evaluate it by accuracy on the test set.
    9. Train a random forest model and evaluate it by accuracy on the test set.
    10. Use GridSearchCV to find the best hyperparameters for the MLP model (five parameter groups).
    11. Save all trained models.
    12. Generate a report summarizing the analysis.
    """
)

def display_run_output(log_lines, run_number, start_index):
    """
    Formats and displays the captured server logs as a clean Markdown report
    within the Jupyter/IPython interface for a specific run.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # grab logs since this run started
    run_logs = log_lines[start_index:]
    
    md_content = f"# LAMBDA Analysis Report — Run {run_number}\n"
    md_content += f"**Generated:** {timestamp}\n\n---\n\n"
    md_content += "```\n"
    md_content += "\n".join(run_logs)
    md_content += "\n```"
    
    display(Markdown(md_content))

def wait_for_completion(log_lines, start_index, timeout=120):
    """
    Wait until LAMBDA signals completion in logs.
    """
    completion_signals = ['Executing result:', 'display_link:', 'Test set accuracy', 'Accuracy on test set']
    elapsed = 0
    while elapsed < timeout:
        run_logs = log_lines[start_index:]
        if any(signal in line for line in run_logs for signal in completion_signals):
            time.sleep(2)
            return True
        time.sleep(1)
        elapsed += 1
    print(f"Warning: timeout after {timeout}s")
    return False


for i in range(2):
    print(f"\n{'='*60}\nRUN {i+1} of 2\n{'='*60}\n")
    
    log_start = len(log_lines)

    job1 = client.submit(
        message=prompt,
        chat_history=[],
        code=None,
        api_name="/chat_streaming"
    )
    updated_chat = job1.result()[1]

    job2 = client.submit(
        chat_history_display=updated_chat,
        code=None,
        api_name="/stream_workflow"
    )
    
    for update in job2:
        pass

    # wait for server logs to show completion instead of fixed sleep
    print("Waiting for analysis to complete...")
    wait_for_completion(log_lines, log_start, timeout=120)

    display_run_output(log_lines, i+1, log_start)
    
    time.sleep(5)

print("All runs complete.")


RUN 1 of 2

Waiting for analysis to complete...


# LAMBDA Analysis Report — Run 1
**Generated:** 2026-03-23 14:27:59

---

```
is_python: True
Executing: Image saved in /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1598389192130397728.png
Executing: Image saved in /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1600368313060394528.png
Executing result: <class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   MedInc                20640 non-null  float64
 1   HouseAge              20640 non-null  float64
 2   AveRooms              20640 non-null  float64
 3   AveBedrms             20640 non-null  float64
 4   Population            20640 non-null  float64
 5   AveOccup              20640 non-null  float64
 6   Latitude              20640 non-null  float64
 7   Longitude             20640 non-null  float64
 8   Affordability_Status  20640 non-null  object
 9   Is_Affordable         20640 non-null  int64
dtypes: float64(8), int64(1), object(1)
memory usage: 1.6+ MB

<Figure size 1200x800 with 2 Axes>
<Figure size 600x400 with 1 Axes>
Fitting 3 folds for each of 48 candidates, totalling 144 fits


California Housing Affordability Dataset Analysis Report
-------------------------------------------------------

1. Dataset Information:
- Shape: (20640, 10)
- Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'Affordability_Status', 'Is_Affordable']
- Target variable: Is_Affordable
- Missing values filled with median for numerical features.

2. Exploratory Data Analysis:
- Correlation heatmap and target distribution plotted.
- Broad feature correlation visible from heatmap.

3. Data Preprocessing:
- Numeric features scaled using StandardScaler.
- Categorical features (if any) one-hot encoded.

4. Model Training and Evaluation (accuracy on test set):
- Logistic Regression accuracy: 0.9998
- Support Vector Machine accuracy: 0.9995
- Multi-layer Perceptron accuracy: 1.0000
- Decision Tree accuracy: 1.0000
- Random Forest accuracy: 1.0000

5. Hyperparameter Tuning:
- Best MLP model found by GridSearchCV with parameters: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'learning_rate': 'constant', 'solver': 'adam'}
- Best MLP test accuracy: 1.0000

6. All models and scaler saved in path:
/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856


     MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  2.344766  0.982143  0.628559  -0.153758   -0.974429 -0.049597  1.052548
1  2.332238 -0.607019  0.327041  -0.263336    0.861439 -0.092512  1.043185
2  1.782699  1.856182  1.155620  -0.049016   -0.820777 -0.025843  1.038503
3  0.932968  1.856182  0.156966  -0.049833   -0.766028 -0.050329  1.038503
4 -0.012881  1.856182  0.344711  -0.032906   -0.759847 -0.085616  1.038503

   Longitude  Affordability_Status_Not Affordable
0  -1.327835                                 True
1  -1.322844                                 True
2  -1.332827                                 True
3  -1.337818                                 True
4  -1.337818                                 True
absolute_path /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1598389192130397728.png
absolute_path /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1600368313060394528.png
display_link: <div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;"><a href="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/decision_tree.pkl" download style="font-weight: bold; color: #007bff;">Download decision_tree.pkl</a></div><div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;"><a href="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/svm.pkl" download style="font-weight: bold; color: #007bff;">Download svm.pkl</a></div><img src="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1598389192130397728.png" style="max-width: 80%;"><div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;"><a href="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/mlp.pkl" download style="font-weight: bold; color: #007bff;">Download mlp.pkl</a></div><div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;"><a href="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/logistic_regression.pkl" download style="font-weight: bold; color: #007bff;">Download logistic_regression.pkl</a></div><div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;"><a href="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/scaler.pkl" download style="font-weight: bold; color: #007bff;">Download scaler.pkl</a></div><div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;"><a href="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/random_forest.pkl" download style="font-weight: bold; color: #007bff;">Download random_forest.pkl</a></div><div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;"><a href="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/best_mlp_gridsearch.pkl" download style="font-weight: bold; color: #007bff;">Download best_mlp_gridsearch.pkl</a></div><img src="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1600368313060394528.png" style="max-width: 80%;">
```


RUN 2 of 2

Waiting for analysis to complete...


# LAMBDA Analysis Report — Run 2
**Generated:** 2026-03-23 14:31:07

---

```
is_python: True
Executing: Image saved in /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1261446052821698272.png
Executing: Image saved in /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1265983737309529824.png
Executing result: <Figure size 1000x800 with 2 Axes>
<Figure size 600x400 with 1 Axes>
Fitting 3 folds for each of 48 candidates, totalling 144 fits


California Housing Affordability Dataset Analysis Report
-------------------------------------------------------

1. Dataset shape: (20640, 9)
2. Target variable: Is_Affordable

3. Model Performance (test accuracy):
- Logistic Regression: 0.8326
- SVM: 0.8639
- MLP: 0.8728
- Decision Tree: 0.8360
- Random Forest: 0.8905

4. Best MLP model parameters from GridSearchCV:
{'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (50, 50), 'learning_rate': 'constant', 'solver': 'adam'}
Best MLP test accuracy: 0.8796

5. All models and scaler saved in: /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856


     MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0 -0.109989  1.295686 -0.229938  -0.224142   -0.739174 -0.065763 -0.674378
1  1.031571  0.424028  0.603860  -0.028408   -0.407977 -0.044816 -0.861521
2 -0.566813  0.265545 -0.147756   0.098100   -0.074109  0.040087  1.084757
3 -0.500348 -0.051422 -0.499797  -0.111813   -0.202315 -0.023257 -0.669700
4  1.396894 -1.002321  0.622457  -0.035448    0.104844 -0.055344  0.953758

   Longitude
0   0.578728
1   0.748399
2  -0.828538
3   0.563757
4  -1.152908
absolute_path /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1261446052821698272.png
absolute_path /Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1265983737309529824.png
display_link: <img src="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1261446052821698272.png" style="max-width: 80%;"><img src="file=/Users/carter/Desktop/Prof.Zheng_Project/LAMBDA/cache/conv_cache/2026-03-23-4679395856/1265983737309529824.png" style="max-width: 80%;">
```

All runs complete.
